# 05 · Enron 419 keyword subset — верификация (плоский JSONL)

Выход: `data/interim/annotated/enron_419_annotated.jsonl`  
Выборка: все уникальные тексты из Enron spam, попавшие под KW_419 (`block_ab/samples/enron_419_unique.jsonl`).

`scenario_family`: `confirmed_advance_fee_scam` | `possible_419_like` | `not_419`


In [ ]:
BATCH_SIZE = 12
SLEEP_SEC = 2.5
MAX_NEW_THIS_RUN = 80
ANNOTATION_MODEL = "openai/gpt-4o-mini"



In [ ]:
import importlib.util, json, os, time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, APIError, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm

def _find_v2_root() -> Path:
    c = Path(globals().get("__vsc_ipynb_file__", globals().get("__file__", "."))).resolve()
    for p in [c, *c.parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists(): return p
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists(): return p
    raise RuntimeError("v2 root")
V2_ROOT = _find_v2_root()
RAW_DIR = V2_ROOT / "data" / "raw" / "collected"
INTERIM = V2_ROOT / "data" / "interim" / "annotated"
BLOCK_AB = INTERIM / "block_ab"
SAMP_DIR = BLOCK_AB / "samples"
OUT_FLAT = INTERIM / "enron_419_annotated.jsonl"
spec = importlib.util.spec_from_file_location("_ann_common", V2_ROOT / "notebooks" / "02_dataset_design" / "_ann_common.py")
ac = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ac)
load_dotenv(V2_ROOT / ".env" if (V2_ROOT / ".env").exists() else None)
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
print(OUT_FLAT)



In [ ]:
enron_all = ac.load_jsonl(RAW_DIR / "enron.jsonl")
enron_spam = [r for r in enron_all if r.get("label") == "spam"]
def build_419():
    return [r for r in ac.dedupe_records_by_text_sha(enron_spam) if ac.is_419_candidate(r.get("text",""))]
sample = ac.ensure_sample(SAMP_DIR / "enron_419_unique.jsonl", build_419)
print("n=", len(sample))



In [ ]:
def wrap(r):
    subj = (r.get("subject") or "").strip()
    body = (r.get("text") or "").strip()
    return [
        {"role": "system", "content": "419/advance-fee verification. JSON: category (confirmed_advance_fee_scam|possible_419_like|not_419), confidence (high|medium|low)."},
        {"role": "user", "content": f"Keyword-filtered candidate.\nSubject:{subj}\n\n{body}"},
    ]
OK = {"confirmed_advance_fee_scam","possible_419_like","not_419"}
def val(d):
    c = d.get("category","possible_419_like")
    if c not in OK: c = "possible_419_like"
    conf = d.get("confidence","low")
    if conf not in {"high","medium","low"}: conf = "low"
    return {"category":c,"confidence":conf}
@retry(retry=retry_if_exception_type((RateLimitError, APIError)), wait=wait_exponential(multiplier=2, min=2, max=45), stop=stop_after_attempt(6))
def call_json(msgs):
    return json.loads(client.chat.completions.create(model=ANNOTATION_MODEL, messages=msgs, response_format={"type":"json_object"}, temperature=0, max_tokens=128).choices[0].message.content)



In [ ]:
idx = ac.load_flat_annotation_index(OUT_FLAT)
pending = [r for r in sample if ac.md5_text_key(r.get("text","")) not in idx]
batch_i = 0
for r in tqdm(pending[:MAX_NEW_THIS_RUN], desc="enron_419"):
    try:
        a = val(call_json(wrap(r)))
    except Exception as e:
        a = val({}); a["_error"] = str(e)[:200]
    ts = datetime.now(timezone.utc).isoformat()
    ex = {"_error": a["_error"]} if "_error" in a else None
    flat = ac.make_flat_record(r, scenario_family=a["category"], annotation_confidence=a["confidence"], annotation_model=ANNOTATION_MODEL, annotated_at=ts, extra=ex)
    ac.append_jsonl(OUT_FLAT, flat)
    batch_i += 1
    if batch_i >= BATCH_SIZE:
        batch_i = 0
        time.sleep(SLEEP_SEC)
print("done", len(ac.load_flat_annotation_index(OUT_FLAT)))

